In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns

In [2]:
df = pd.read_csv('cleaned_sms_data.csv')
df.head()

,target,text,num_characters,num_words,num_sentences,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",111,24,2,go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,29,8,2,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,155,37,2,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,49,13,1,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",61,15,1,nah think goe usf live around though


In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=4000, stop_words='english')

In [4]:
df['transformed_text'].isnull().sum()

np.int64(9)

In [5]:
# Drop rows where the transformed text is null
df = df.dropna(subset=['transformed_text'])

In [6]:
df['transformed_text'].isnull().sum()

np.int64(0)

In [36]:
X = tfidf.fit_transform(df['transformed_text']).toarray()

In [19]:
X.shape

(5160, 1000)

In [20]:
y = df['target'].values

In [21]:
y.shape

(5160,)

In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [39]:
gnb = GaussianNB()
mnb = MultinomialNB()
bnb = BernoulliNB()

In [40]:
gnb.fit(X_train,y_train)
y_pred1 = gnb.predict(X_test)
print(accuracy_score(y_test,y_pred1)*100)
print(confusion_matrix(y_test,y_pred1))
print(precision_score(y_test,y_pred1)*100)

84.78682170542635
[[778 131]
 [ 26  97]]
42.54385964912281


In [41]:
mnb.fit(X_train,y_train)
y_pred2 = mnb.predict(X_test)
print(accuracy_score(y_test,y_pred2)*100)
print(confusion_matrix(y_test,y_pred2))
print(precision_score(y_test,y_pred2)*100)

96.80232558139535
[[909   0]
 [ 33  90]]
100.0


In [42]:
bnb.fit(X_train,y_train)
y_pred3 = bnb.predict(X_test)
print(accuracy_score(y_test,y_pred3)*100) 
print(confusion_matrix(y_test,y_pred3))
print(precision_score(y_test,y_pred3)*100)

96.60852713178295
[[903   6]
 [ 29  94]]
94.0


In [ ]:
## ttfidf --> mnb 

## Use Another Model to check the Accuracy

In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [50]:
lr = LogisticRegression(solver='liblinear', penalty='l1')
svc = SVC(kernel='sigmoid', gamma=1.0)
dt = DecisionTreeClassifier(max_depth=5)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(n_estimators=50, random_state=2)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)
gbc = GradientBoostingClassifier(n_estimators=50, random_state=2)
xgb = XGBClassifier(n_estimators=50, random_state=2)

In [54]:
clfs = {
    'LR': lr,
    'SVC': svc,
    'DT': dt,
    'KNN': knn,
    'RF': rf,
    'AdaBoost': abc,
    'Bagging': bc,
    'ExtraTrees': etc,
    'GBDT': gbc,
    'XGB': xgb
}

In [53]:
# 2. Loop through each model, train, and collect scores
accuracy_scores = []
precision_scores = []

print("Model | Accuracy (%) | Precision (%)")
print("-" * 38)

for name, clf in clfs.items():
    # Train the model
    clf.fit(X_train, y_train)
    # Predict on test data
    y_pred = clf.predict(X_test)
    
    # Calculate scores
    acc = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred) * 100
    
    # Print individual performance live
    print(f"{name:<10} | {acc:.2f}% | {prec:.2f}%")

Model | Accuracy (%) | Precision (%)
--------------------------------------
LR         | 95.83% | 92.55%


C:\Users\SAHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\SAHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


SVC        | 97.38% | 95.28%
DT         | 93.31% | 87.50%
KNN        | 92.44% | 97.87%
RF         | 97.48% | 98.99%
AdaBoost   | 91.57% | 92.86%
Bagging    | 96.41% | 85.25%
ExtraTrees | 97.38% | 96.15%
GBDT       | 95.16% | 95.06%
XGB        | 96.71% | 96.84%


In [57]:
import pickle
pickle.dump(tfidf,open('vectorizer.pkl', 'wb'))
pickle.dump(mnb, open('model.pkl','wb'))

In [68]:

# 1. Load the vectorizer and model back into memory
loaded_tfidf = pickle.load(open('vectorizer.pkl', 'rb'))
loaded_model = pickle.load(open('model.pkl', 'rb'))

# 2. Define a fake test message
test_msg ="URGENT CALL ME NOW. THIS IS NOT A JOKE."

# 3. Transform and predict
transformed_msg = loaded_tfidf.transform([test_msg]).toarray()
prediction = loaded_model.predict(transformed_msg)[0]

# 4. Print the result (1 = Spam, 0 = Ham)
print("Prediction code works! Result:", "SPAM" if prediction == 1 else "HAM")


Prediction code works! Result: HAM
